# Demonstration of ValidatorAgent and ManagerAgent
This notebook tests the new architecture by simulating task execution and validation using a mock labor agent.

In [1]:
import os
import random
import sys
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

from agentlite.agents.ValidatorAgent import ValidatorAgent
from agentlite.agents.ManagerAgent import ManagerAgent
from agentlite.agents.BaseAgent import BaseAgent
from agentlite.commons import TaskPackage
from agentlite.llm.agent_llms import get_llm_backend
from agentlite.llm.LLMConfig import LLMConfig

# Initialize LLM Backend
Set up the LLM backend using LLMConfig and get_llm_backend.

In [2]:
llm_config_dict = {
    "llm_name": "gemma3:4b", # "deepseek-r1:8b"
    "temperature": 0.7, "provider": 
    "ollama", "base_url": 
    "http://localhost:11434"}
llm_config = LLMConfig(llm_config_dict)
llm = get_llm_backend(llm_config)

In [3]:
from agentlite.actions import BaseAction, FinishAct
from agentlite.actions.InnerActions import INNER_ACT_KEY
from agentlite.commons import AgentAct

class AnswerQuestion(BaseAction):
    def __init__(self):
        super().__init__(
            action_name="AnswerQuestion",
            action_desc="Answer a given question based on knowledge",
            params_doc={
                "question": "The question to be answered"
            }
        )
    
    def __call__(self, question: str) -> str:
        # In a real scenario, this would call an API or database
        # For demo purposes, we're just returning a mock answer
        if "capital" in question.lower() and "france" in question.lower():
            return "Paris is the capital of France."
        elif "weather" in question.lower():
            return "I'm sorry, I don't have real-time weather data."
        else:
            return "I don't know the answer to that question."

class MockLaborAgent(BaseAgent):
    def __init__(self, llm):
        super().__init__(
            name="MockLaborAgent",
            role="Provide answers to questions based on knowledge.",
            actions=[AnswerQuestion(), FinishAct],
            llm=llm,
            reasoning_type="act"  # Using 'act' reasoning which adds FinishAct automatically
        )

    def respond(self, task_pkg: TaskPackage, **kwargs):
        question = task_pkg.instruction
        action = self.actions[0]  # AnswerQuestion action
        answer = action(question=question)
        return answer

mock_labor_agent = MockLaborAgent(llm)

In [4]:
manager_agent = ManagerAgent(
    llm=llm,
    name="TestManagerAgent",
    role="Manage tasks and validate responses.",
    TeamAgents=[mock_labor_agent]
)

# Define and Execute Test Task
Define a test task using TaskPackage, simulate task execution, and print the final response.

In [5]:
# Let's create another test with a potentially incorrect answer
class MockIncorrectAgent(BaseAgent):
    def __init__(self, llm):
        super().__init__(
            name="MockIncorrectAgent",
            role="Sometimes provides incorrect answers to questions.",
            actions=[],
            llm=llm
        )

    def respond(self, task_pkg: TaskPackage, **kwargs):
        # Deliberately return an incorrect answer
        # randomly
        if random.choice([True, False]):
            return "The capital of France is Lyon."  # Incorrect answer
        return "The capital of France is Paris."  # Correct answer

# Create the incorrect agent
mock_incorrect_agent = MockIncorrectAgent(llm)

# Add it to the manager's team
manager_agent.add_member(mock_incorrect_agent)

# Create a new test task
test_task = TaskPackage(
    instruction="What is the capital of France?",
    task_creator="Tester",
    task_executor="MockIncorrectAgent"
)

# Create an action that targets the incorrect agent
agent_act = AgentAct(
    name="MockIncorrectAgent",
    params={"Task": "What is the capital of France?"}
)

print("\n\nTesting with incorrect answer:")
print("Executing action on incorrect agent...")
response2 = manager_agent.forward(test_task, agent_act)

print("\nValidation result:\n", response2)



Testing with incorrect answer:
Executing action on incorrect agent...
Agent MockIncorrectAgent receives the following TaskPackage:
[
	Task ID: 075ce809-a665-4f85-82a8-947dabf9fe42
	Instruction: What is the capital of France?
]
====MockIncorrectAgent starts execution on TaskPackage 075ce809-a665-4f85-82a8-947dabf9fe42====
Agent MockIncorrectAgent takes 0-step Action:
{
	name: Think
	params: {'response': 'The capital of France is Paris. This is a well-known fact and a standard question to test basic knowledge.'}
}
Observation: OK
Agent MockIncorrectAgent takes 1-step Action:
{
	name: Finish
	params: {'response': 'The capital of France is Paris.'}
}
Observation: The capital of France is Paris.
=========MockIncorrectAgent finish execution. TaskPackage[ID:075ce809-a665-4f85-82a8-947dabf9fe42] status:
[
	completion: completed
	answer: The capital of France is Paris.
]
Action and Observation added to Agent Labor agent MockIncorrectAgent provided initial response (attempt 1) memory
&&& Valid